# Building the Estimation Panel: Merging Productivity and Trade-Agreement Data

This notebook merges the McMillan–Rodrik decomposition outputs
(`notebooks/03_clean_ggdc_productivity.ipynb`) with the trade-agreement panel
(`notebooks/01_clean_trade_agreements.ipynb` / `02_clean_cepii_gravity.ipynb`) into
the single analysis-ready dataset methodology doc §6.2 step 6 and §6.4 are built
around — the last step before actually running the staggered-DiD regressions.

**Design choices carried over from prior work, not re-decided here:**

- **Sector-level, not country-level, as the primary grain** — this project's
  research question is about sectoral productivity specifically (see the design
  discussion recorded in `docs/developing-country-trade-productivity.md` §6.3).
  Country-level totals are kept too, as a `broad_sector == "Total"` pseudo-sector
  row alongside Agriculture/Manufacturing/Services, rather than a separate file —
  the same convention the raw GGDC source itself uses.
- **Interval panels (3-year and 5-year), not annual or full-period** — both widths
  are merged into their own output file (`estimation_panel_interval3.csv`,
  `estimation_panel_interval5.csv`), matching notebook 03 §12b's reasoning:
  annual is too noisy as a regression outcome, full-period leaves no panel
  structure for a staggered estimator.
- **Trade-agreement-side control variables are attached at `year0`** (the
  beginning of each interval), not `year1` or an average — using values the
  treatment itself could have affected as controls risks "bad control" bias, so
  pre-determined (start-of-interval) values are used throughout except for
  `reciprocal`, which is attached at *both* `year0` and `year1` to determine
  whether the interval is genuinely pre-treatment, post-treatment, or straddles
  the switch.

**What this notebook does, in order:**

1. Load the four sources: the trade-agreement country-year panel, and the
   sector-level + country-level decomposition files, for both interval widths.
2. Fold the country-level totals into the sector-level file as a `"Total"`
   pseudo-sector, producing one long-format table per interval width.
3. Merge in the trade-agreement variables at `year0` (controls) and `reciprocal`
   at `year1` (to classify each interval's treatment status).
4. Classify every interval as `never_reciprocal`, `always_reciprocal`, or
   `switched_during_interval` — and confirm directly, not assume, that the fourth
   logical case (a reversion) never occurs, consistent with `Reciprocal_c,t`'s
   confirmed-absorbing status from notebook 01.
5. Validate the merge (coverage, no silent row loss, spot-check against a known
   case — South Africa's 2000 EU TDCA entry).
6. Save the two estimation panels.

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

In [2]:
REPO_ROOT = Path("..")
TRADE_PANEL = REPO_ROOT / "data" / "processed" / "trade_agreements_with_exposure_country_year.csv"
DECOMP_DIR = REPO_ROOT / "data" / "processed"
OUT_DIR = REPO_ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)

TRADE_PANEL

WindowsPath('../data/processed/trade_agreements_with_exposure_country_year.csv')

## 2. Load the sources

In [3]:
trade = pd.read_csv(TRADE_PANEL)

AFRICAN = {
    "DZA": "Algeria", "AGO": "Angola", "BWA": "Botswana", "BFA": "Burkina Faso",
    "CMR": "Cameroon", "EGY": "Egypt", "SWZ": "Eswatini", "ETH": "Ethiopia",
    "GHA": "Ghana", "KEN": "Kenya", "LSO": "Lesotho", "MWI": "Malawi",
    "MUS": "Mauritius", "MAR": "Morocco", "MOZ": "Mozambique", "NAM": "Namibia",
    "NGA": "Nigeria", "RWA": "Rwanda", "SEN": "Senegal", "SLE": "Sierra Leone",
    "ZAF": "South Africa", "TZA": "United Republic of Tanzania", "UGA": "Uganda",
    "ZMB": "Zambia",
}

print(f"Trade panel: {len(trade):,} rows, {trade['iso3'].nunique()} countries, "
      f"{trade['year'].min()}-{trade['year'].max()}")
trade.head(3)

Trade panel: 1,632 rows, 24 countries, 1950-2017


,iso3,year,country_exists,depth_score,agreement_type,reciprocal,fta_or_deeper,nonreciprocal_only,n_northern_reporting,n_northern_with_agreement,trade_total_northern,gdp_total_northern,trade_reciprocal_northern,gdp_reciprocal_northern,n_reciprocal_partners,n_partners_trade_available,reciprocal_trade_share,reciprocal_gdp_share,gdp_african,gdpcap_african,gdp_ppp_african,gdpcap_ppp_african,pop_african,pop_pwt_african,gdp_ppp_pwt_african,wto_african,gatt_african
0,AGO,1950,True,0.0,0_none,0.0,0.0,0.0,28.0,0.0,109650.0,4.391497e+08,0.0,0.0,0.0,14.0,0.0,0.0,NaN,NaN,NaN,NaN,4117.617,NaN,NaN,0.0,0.0
1,AGO,1951,True,0.0,0_none,0.0,0.0,0.0,28.0,0.0,152450.0,5.087046e+08,0.0,0.0,0.0,15.0,0.0,0.0,NaN,NaN,NaN,NaN,4173.095,NaN,NaN,0.0,0.0
2,AGO,1952,True,0.0,0_none,0.0,0.0,0.0,28.0,0.0,154550.0,5.641776e+08,0.0,0.0,0.0,19.0,0.0,0.0,NaN,NaN,NaN,NaN,4232.095,NaN,NaN,0.0,0.0


## 3. Fold country-level totals into the sector-level table as `"Total"`

For each interval width, the country-level file's `Y0`/`Y1` (economy-wide
productivity) become that width's `"Total"` row's own `productivity_0`/`productivity_1`
— i.e., `"Total"` is treated as a fourth sector whose "employment share" of the
whole economy is trivially 1.0, the same convention the raw GGDC source uses for
its own `Total` row alongside the 9 sub-sectors.

In [4]:
def load_and_stack(width):
    country = pd.read_csv(DECOMP_DIR / f"mcmillan_rodrik_decomposition_interval{width}.csv")
    sector = pd.read_csv(DECOMP_DIR / f"mcmillan_rodrik_decomposition_interval{width}_by_sector.csv")

    total_rows = country.copy()
    total_rows["broad_sector"] = "Total"
    total_rows["employment_share_0"] = 1.0
    total_rows["employment_share_1"] = 1.0
    total_rows["productivity_0"] = total_rows["Y0"]
    total_rows["productivity_1"] = total_rows["Y1"]

    cols = ["iso3", "year0", "year1", "broad_sector", "employment_share_0",
            "employment_share_1", "productivity_0", "productivity_1",
            "within", "structural_change", "short_series"]
    stacked = pd.concat([sector[cols], total_rows[cols]], ignore_index=True)

    # Country-level context columns, attached to every row in the group (sector
    # rows included) -- explicitly prefixed "country_" so a user filtering to
    # sector rows doesn't mistake these for that sector's own change.
    ctx = country[["iso3", "year0", "year1", "Y0", "Y1", "actual_change", "decomposed_change"]].rename(
        columns={"Y0": "country_productivity_0", "Y1": "country_productivity_1",
                 "actual_change": "country_actual_change",
                 "decomposed_change": "country_decomposed_change"}
    )
    stacked = stacked.merge(ctx, on=["iso3", "year0", "year1"], how="left")
    stacked["country"] = stacked["iso3"].map(AFRICAN)
    return stacked


stacked3 = load_and_stack(3)
stacked5 = load_and_stack(5)
print(f"interval3: {len(stacked3):,} rows ({stacked3['broad_sector'].value_counts().to_dict()})")
print(f"interval5: {len(stacked5):,} rows ({stacked5['broad_sector'].value_counts().to_dict()})")
stacked3.head(4)

interval3: 1,404 rows ({'Agriculture': 351, 'Manufacturing': 351, 'Services': 351, 'Total': 351})
interval5: 804 rows ({'Agriculture': 201, 'Manufacturing': 201, 'Services': 201, 'Total': 201})


,iso3,year0,year1,broad_sector,employment_share_0,employment_share_1,productivity_0,productivity_1,within,structural_change,short_series,country_productivity_0,country_productivity_1,country_actual_change,country_decomposed_change,country
0,AGO,2002,2005,Agriculture,0.369608,0.384126,129.449896,131.681268,0.824732,1.911735,False,737.202974,825.63864,88.435666,88.435666,Angola
1,AGO,2002,2005,Manufacturing,0.087696,0.086278,4184.590579,5061.849457,76.932400,-7.177230,False,737.202974,825.63864,88.435666,88.435666,Angola
2,AGO,2002,2005,Services,0.542696,0.529596,594.041548,638.841667,24.312847,-8.368817,False,737.202974,825.63864,88.435666,88.435666,Angola
3,AGO,2005,2008,Agriculture,0.384126,0.445860,131.681268,118.794335,-4.950200,7.333695,False,825.638640,935.31629,109.677650,109.677650,Angola


## 4. Merge in the trade-agreement variables

Controls at `year0` (pre-determined, avoiding "bad control" bias from using
values the treatment could itself have influenced); `reciprocal` also pulled at
`year1` specifically to classify the interval's treatment status below.

In [5]:
CONTROL_COLS = [
    "country_exists", "depth_score", "agreement_type", "reciprocal", "fta_or_deeper",
    "nonreciprocal_only", "reciprocal_trade_share", "reciprocal_gdp_share",
    "gdp_african", "gdpcap_african", "pop_african", "wto_african", "gatt_african",
]


def merge_trade(panel):
    t0 = trade[["iso3", "year"] + CONTROL_COLS].rename(
        columns={**{c: f"{c}_pre" for c in CONTROL_COLS}, "year": "year0"}
    )
    t1 = trade[["iso3", "year", "reciprocal", "country_exists"]].rename(
        columns={"reciprocal": "reciprocal_post", "country_exists": "country_exists_post",
                 "year": "year1"}
    )
    panel = panel.merge(t0, on=["iso3", "year0"], how="left")
    panel = panel.merge(t1, on=["iso3", "year1"], how="left")
    return panel


estimation3 = merge_trade(stacked3)
estimation5 = merge_trade(stacked5)

for name, df in [("interval3", estimation3), ("interval5", estimation5)]:
    unmatched = df["country_exists_pre"].isna().sum() + df["country_exists_post"].isna().sum()
    print(f"{name}: {len(df):,} rows, unmatched year0/year1 lookups: {unmatched} (expect 0 -- "
          f"every interval boundary year should find a trade-panel row, even if that "
          f"row itself says the country didn't exist yet)")

interval3: 1,404 rows, unmatched year0/year1 lookups: 0 (expect 0 -- every interval boundary year should find a trade-panel row, even if that row itself says the country didn't exist yet)
interval5: 804 rows, unmatched year0/year1 lookups: 0 (expect 0 -- every interval boundary year should find a trade-panel row, even if that row itself says the country didn't exist yet)


## 5. Classify treatment status over each interval

`Reciprocal_c,t` was confirmed empirically absorbing in notebook 01 (never reverts
from 1 to 0) — that should still hold at the interval level, but this is checked
directly rather than assumed to carry over automatically.

In [6]:
def classify_treatment(panel):
    conditions = [
        (panel["reciprocal_pre"] == 0) & (panel["reciprocal_post"] == 0),
        (panel["reciprocal_pre"] == 1) & (panel["reciprocal_post"] == 1),
        (panel["reciprocal_pre"] == 0) & (panel["reciprocal_post"] == 1),
        (panel["reciprocal_pre"] == 1) & (panel["reciprocal_post"] == 0),
    ]
    choices = ["never_reciprocal", "always_reciprocal", "switched_during_interval",
               "IMPOSSIBLE_reverted"]
    panel = panel.copy()
    panel["treatment_status"] = np.select(conditions, choices, default="missing_reciprocal_data")
    panel["reciprocal_interval"] = panel["treatment_status"].isin(
        ["always_reciprocal", "switched_during_interval"]
    ).astype(int)
    return panel


estimation3 = classify_treatment(estimation3)
estimation5 = classify_treatment(estimation5)

for name, df in [("interval3", estimation3), ("interval5", estimation5)]:
    n_reverted = (df["treatment_status"] == "IMPOSSIBLE_reverted").sum()
    assert n_reverted == 0, f"{name}: found {n_reverted} reverted intervals -- investigate before proceeding"
    print(f"{name} treatment_status counts:")
    print(df["treatment_status"].value_counts().to_string())
    print()
print("Confirmed: zero reverted intervals in either file -- the absorbing property "
      "holds at interval resolution too, not just annually.")

interval3 treatment_status counts:
treatment_status
never_reciprocal            1096
always_reciprocal            236
switched_during_interval      64
missing_reciprocal_data        8

interval5 treatment_status counts:
treatment_status
never_reciprocal            620
always_reciprocal           116
switched_during_interval     64
missing_reciprocal_data       4

Confirmed: zero reverted intervals in either file -- the absorbing property holds at interval resolution too, not just annually.


**`missing_reciprocal_data` rows are a real, already-understood data feature, not
a merge failure** — they occur only where the country did not yet exist as a
recognized state at `year0` or `year1` (e.g. Tanzania's 1960–1963 interval, before
the 1964 Tanganyika–Zanzibar union), matching the `NoCty` convention from notebook
01. Confirmed by inspecting the affected rows directly below.

In [7]:
missing = estimation3[estimation3["treatment_status"] == "missing_reciprocal_data"]
missing[["iso3", "year0", "year1", "country_exists_pre", "country_exists_post"]].drop_duplicates()

,iso3,year0,year1,country_exists_pre,country_exists_post
861,TZA,1960,1963,False,False
864,TZA,1963,1966,False,True


## 6. Validate against a known case: South Africa's 2000 EU TDCA entry

South Africa's TDCA entered into force in 2000, which should fall inside a
`switched_during_interval` bin at both interval widths, with the within/
structural-change values for that bin matching what notebook 03 already computed
and validated.

In [8]:
check = estimation3[(estimation3["iso3"] == "ZAF") & (estimation3["year0"].between(1996, 2002))]
check[["year0", "year1", "broad_sector", "treatment_status", "reciprocal_interval",
       "within", "structural_change"]].sort_values(["year0", "broad_sector"])

,year0,year1,broad_sector,treatment_status,reciprocal_interval,within,structural_change
981,1996,1999,Agriculture,never_reciprocal,0,-0.119228,-0.156225
982,1996,1999,Manufacturing,never_reciprocal,0,-2.021485,-0.965780
983,1996,1999,Services,never_reciprocal,0,-1.560667,1.400436
1380,1996,1999,Total,never_reciprocal,0,-3.701380,0.278432
984,1999,2002,Agriculture,switched_during_interval,1,-0.111776,0.376967
985,1999,2002,Manufacturing,switched_during_interval,1,0.719815,2.315239
986,1999,2002,Services,switched_during_interval,1,12.871550,-3.810484
1381,1999,2002,Total,switched_during_interval,1,13.479590,-1.118278
987,2002,2005,Agriculture,always_reciprocal,1,2.271575,-2.474329
988,2002,2005,Manufacturing,always_reciprocal,1,0.805006,-0.395666


## 7. Save the estimation panels

In [9]:
for name, df in [("interval3", estimation3), ("interval5", estimation5)]:
    out = OUT_DIR / f"estimation_panel_{name}.csv"
    df.to_csv(out, index=False)
    print(f"Saved {len(df):,} rows, {df.shape[1]} columns -> {out}")

Saved 1,404 rows, 33 columns -> ..\data\processed\estimation_panel_interval3.csv
Saved 804 rows, 33 columns -> ..\data\processed\estimation_panel_interval5.csv


## 8. Limitations and next steps

- **`switched_during_interval` bins mix pre- and post-treatment dynamics** — the
  decomposition components for these intervals reflect a country that was not yet
  reciprocal-treated for part of the window and was treated for the rest, so they
  shouldn't be pooled uninspected with clean `never_reciprocal`/`always_reciprocal`
  bins in a regression. `treatment_status` is carried as an explicit column
  precisely so this can be handled deliberately (e.g. dropped, or treated as its
  own category) rather than silently averaged over.
- **Controls are pre-determined (`year0`) by design** — this avoids "bad control"
  bias, but means e.g. `gdp_african_pre` describes the country's GDP at the
  *start* of the interval the outcome is measured over, not concurrent with it.
- **Only a subset of the trade panel's columns were carried over** as `_pre`
  controls (see §4) — the full set (e.g. `n_northern_reporting`, GDP/trade totals
  with all Northern partners) remains in `trade_agreements_with_exposure_country_year.csv`
  if needed later.
- **The `short_series` flag (Angola, Sierra Leone at 5-year resolution) is
  inherited as-is from notebook 03** — still not excluded here, per the
  recommendation already recorded in `docs/developing-country-trade-productivity.md`
  §6.3 to keep them in the baseline sample and treat exclusion as a robustness
  check.
- **Next step**: run methodology doc §6.4's staggered-adoption-robust estimator
  (Callaway–Sant'Anna or Sellner–Yotov's ETWFE) with `within`/`structural_change`
  (filtered to `broad_sector != "Total"` for the primary sector-level
  specification, or `== "Total"` for the country-level companion) as the outcome
  and `reciprocal_interval` as the treatment, dropping or separately handling
  `switched_during_interval` rows as appropriate for the chosen estimator.